In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('indian_flights.csv')
df = data.copy()

In [3]:
df.head()

,flight_date,airline,source,destination,route,departure_time,total_stops,duration_mins,cabin_class,days_left,is_weekend,price
0,2025-04-29,Air India,Mumbai,Goa,BOM-GOI,Evening (6PM-12AM),1-stop,344,Economy,83,0,3010
1,2024-01-16,SpiceJet,Hyderabad,Cochin,HYD-COK,Evening (6PM-12AM),1-stop,335,Economy,4,0,5353
2,2025-12-16,Akasa Air,Hyderabad,Delhi,HYD-DEL,Early Morning (12AM-6AM),non-stop,135,Economy,82,0,4795
3,2024-08-04,Air India,Mumbai,Bengaluru,BOM-BLR,Morning (6AM-12PM),1-stop,298,Economy,43,1,4121
4,2024-09-12,Akasa Air,Hyderabad,Cochin,HYD-COK,Early Morning (12AM-6AM),non-stop,117,Economy,45,0,3066


In [4]:
df.columns

Index(['flight_date', 'airline', 'source', 'destination', 'route',
       'departure_time', 'total_stops', 'duration_mins', 'cabin_class',
       'days_left', 'is_weekend', 'price'],
      dtype='str')

In [5]:
df.columns = df.columns.str.lower()

In [6]:
def checkUnique(col):
    print(df[col].unique())

def checkValueCounts(col):
    print(df[col].value_counts(dropna = False))

def toNumeric(col):
    df[col] = pd.to_numeric(df[col],errors="coerce")

In [7]:
df = df.drop('route', axis=1) 

In [8]:
checkValueCounts('total_stops')

total_stops
non-stop    324965
1-stop      149857
2-stops      25178
Name: count, dtype: int64


In [9]:
df['total_stops'] = df['total_stops'].replace(
    {'non-stop': 0,
     '1-stop': 1,
     '2-stops': 2,})

In [10]:
checkValueCounts('total_stops')

total_stops
0    324965
1    149857
2     25178
Name: count, dtype: int64


In [11]:
toNumeric('total_stops')

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   flight_date     500000 non-null  str  
 1   airline         500000 non-null  str  
 2   source          500000 non-null  str  
 3   destination     500000 non-null  str  
 4   departure_time  500000 non-null  str  
 5   total_stops     500000 non-null  int64
 6   duration_mins   500000 non-null  int64
 7   cabin_class     500000 non-null  str  
 8   days_left       500000 non-null  int64
 9   is_weekend      500000 non-null  int64
 10  price           500000 non-null  int64
dtypes: int64(5), str(6)
memory usage: 42.0 MB


In [13]:
df.select_dtypes(include=['object']).columns

/tmp/ipykernel_39101/549554427.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object']).columns


Index(['flight_date', 'airline', 'source', 'destination', 'departure_time',
       'cabin_class'],
      dtype='str')

In [14]:
df.head()

,flight_date,airline,source,destination,departure_time,total_stops,duration_mins,cabin_class,days_left,is_weekend,price
0,2025-04-29,Air India,Mumbai,Goa,Evening (6PM-12AM),1,344,Economy,83,0,3010
1,2024-01-16,SpiceJet,Hyderabad,Cochin,Evening (6PM-12AM),1,335,Economy,4,0,5353
2,2025-12-16,Akasa Air,Hyderabad,Delhi,Early Morning (12AM-6AM),0,135,Economy,82,0,4795
3,2024-08-04,Air India,Mumbai,Bengaluru,Morning (6AM-12PM),1,298,Economy,43,1,4121
4,2024-09-12,Akasa Air,Hyderabad,Cochin,Early Morning (12AM-6AM),0,117,Economy,45,0,3066


In [15]:
cat_cols = ['airline', 'source', 'destination','cabin_class']
df = pd.get_dummies(df, columns=cat_cols, drop_first=False,dtype=int) 
# multicollinearity is not a concern for tree-based models, so we can keep all the dummy variables.


In [16]:
df.columns

Index(['flight_date', 'departure_time', 'total_stops', 'duration_mins',
       'days_left', 'is_weekend', 'price', 'airline_Air India',
       'airline_Akasa Air', 'airline_IndiGo', 'airline_SpiceJet',
       'airline_Vistara', 'source_Bengaluru', 'source_Chennai',
       'source_Cochin', 'source_Delhi', 'source_Hyderabad', 'source_Kolkata',
       'source_Mumbai', 'destination_Ahmedabad', 'destination_Bengaluru',
       'destination_Cochin', 'destination_Delhi', 'destination_Goa',
       'destination_Hyderabad', 'destination_Mumbai', 'cabin_class_Business',
       'cabin_class_Economy'],
      dtype='str')

In [17]:
df['flight_date'] = pd.to_datetime(df['flight_date'],format='%Y-%m-%d')

In [18]:
df['flight_year'] = df['flight_date'].dt.year
df['flight_month'] = df['flight_date'].dt.month
df['flight_day'] = df['flight_date'].dt.day

In [19]:
df.head()

,flight_date,departure_time,total_stops,duration_mins,days_left,is_weekend,price,airline_Air India,airline_Akasa Air,airline_IndiGo,...,destination_Cochin,destination_Delhi,destination_Goa,destination_Hyderabad,destination_Mumbai,cabin_class_Business,cabin_class_Economy,flight_year,flight_month,flight_day
0,2025-04-29,Evening (6PM-12AM),1,344,83,0,3010,1,0,0,...,0,0,1,0,0,0,1,2025,4,29
1,2024-01-16,Evening (6PM-12AM),1,335,4,0,5353,0,0,0,...,1,0,0,0,0,0,1,2024,1,16
2,2025-12-16,Early Morning (12AM-6AM),0,135,82,0,4795,0,1,0,...,0,1,0,0,0,0,1,2025,12,16
3,2024-08-04,Morning (6AM-12PM),1,298,43,1,4121,1,0,0,...,0,0,0,0,0,0,1,2024,8,4
4,2024-09-12,Early Morning (12AM-6AM),0,117,45,0,3066,0,1,0,...,1,0,0,0,0,0,1,2024,9,12


In [20]:
df[['departure_time']].head()

,departure_time
0,Evening (6PM-12AM)
1,Evening (6PM-12AM)
2,Early Morning (12AM-6AM)
3,Morning (6AM-12PM)
4,Early Morning (12AM-6AM)


In [21]:
checkValueCounts('departure_time')

departure_time
Evening (6PM-12AM)          125263
Afternoon (12PM-6PM)        125146
Early Morning (12AM-6AM)    124841
Morning (6AM-12PM)          124750
Name: count, dtype: int64


In [22]:
df = pd.get_dummies(df, columns=['departure_time'], drop_first=False, dtype=int)

In [23]:
df.columns

Index(['flight_date', 'total_stops', 'duration_mins', 'days_left',
       'is_weekend', 'price', 'airline_Air India', 'airline_Akasa Air',
       'airline_IndiGo', 'airline_SpiceJet', 'airline_Vistara',
       'source_Bengaluru', 'source_Chennai', 'source_Cochin', 'source_Delhi',
       'source_Hyderabad', 'source_Kolkata', 'source_Mumbai',
       'destination_Ahmedabad', 'destination_Bengaluru', 'destination_Cochin',
       'destination_Delhi', 'destination_Goa', 'destination_Hyderabad',
       'destination_Mumbai', 'cabin_class_Business', 'cabin_class_Economy',
       'flight_year', 'flight_month', 'flight_day',
       'departure_time_Afternoon (12PM-6PM)',
       'departure_time_Early Morning (12AM-6AM)',
       'departure_time_Evening (6PM-12AM)',
       'departure_time_Morning (6AM-12PM)'],
      dtype='str')

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 34 columns):
 #   Column                                   Non-Null Count   Dtype         
---  ------                                   --------------   -----         
 0   flight_date                              500000 non-null  datetime64[us]
 1   total_stops                              500000 non-null  int64         
 2   duration_mins                            500000 non-null  int64         
 3   days_left                                500000 non-null  int64         
 4   is_weekend                               500000 non-null  int64         
 5   price                                    500000 non-null  int64         
 6   airline_Air India                        500000 non-null  int64         
 7   airline_Akasa Air                        500000 non-null  int64         
 8   airline_IndiGo                           500000 non-null  int64         
 9   airline_SpiceJet                     

In [25]:
df.drop('flight_date',axis=1,inplace=True)

/tmp/ipykernel_39101/2050363305.py:1: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop('flight_date',axis=1,inplace=True)


In [26]:
df.to_csv("indian_flights_cleaned.csv")